In [1]:
!nvidia-smi

Tue Sep 15 05:39:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q ultralytics roboflow dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9

In [3]:
from roboflow import Roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.152 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 7035.9/8062.4 GB disk)


In [4]:
import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("dagshub")
from ultralytics import settings

dagshub.auth.add_app_token(token)
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Soccer-Tracking')

# Optional: Set the experiment name
mlflow.set_experiment("tunning")


settings.update({'mlflow': True})

Accessing as AgabaEmbedded

Initialized MLflow to track repo "AgabaEmbedded/Soccer-Tracking"

Repository AgabaEmbedded/Soccer-Tracking initialized!

In [5]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="9VCfGRMmzvi61B4c9qu5")
project = rf.workspace("agabaembedded").project("soccer-tracking-large")
version = project.version(2)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Soccer-Tracking-Large-2 in yolo26:: 100%|██████████| 17105/17105 [00:05<00:00, 2901.90it/s]


In [6]:
!yolo train \
model=yolo26m.pt \
name=Large-Images-3Classes \
data=/kaggle/working/Soccer-Tracking-Large-2/data.yaml \
epochs=150 \
imgsz=1088,1920 \
device=0,1 \
plots=True \
batch=4 \
time=6 \
resume=True \
pretrained=True \
save=True \
deterministic=False \
optimizer=SGD \
cos_lr=True \
lr0=0.00683 \
lrf=0.00933 \
momentum=0.88752 \
weight_decay=0.00049 \
warmup_epochs=4.35648 \
warmup_momentum=0.55489 \
box=3 \
cls=0.83514 \
dfl=1.20718 \
hsv_h=0.0 \
hsv_s=0.0 \
hsv_v=0.0 \
degrees=0.0 \
translate=0.0 \
scale=0.2 \
shear=0.0 \
perspective=0.0 \
flipud=0.0 \
fliplr=0.0 \
bgr=0.0 \
mosaic=0.0 \
mixup=0.0 \
cutmix=0.0 \
copy_paste=0.0

WARNING ⚠️ model 'yolo26m.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.152 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=3, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.83514, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/Soccer-Tracking-Large-2/data.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.20718, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1

In [7]:
import os
import dagshub
import mlflow
from mlflow.tracking import MlflowClient

# 1. Initialize your DagsHub connection
dagshub.init(repo_owner='AgabaEmbedded', repo_name='Soccer-Tracking')

# 2. Set the experiment name to keep things organized
mlflow.set_experiment("Large Images")

# 3. Path to your completed weights files
local_weights_path = "/kaggle/working/runs/detect/Large-Images-3Classes/weights/best.pt"
last_weights_path = "/kaggle/working/runs/detect/Large-Images-3Classes/weights/last.pt"

if not os.path.exists(local_weights_path):
    raise FileNotFoundError(f"Could not find weights at {local_weights_path}.")
if not os.path.exists(last_weights_path):
    raise FileNotFoundError(f"Could not find weights at {last_weights_path}.")

# 4. Start a manual MLflow run to log and register the model
with mlflow.start_run(run_name="model_upload") as run:
    
    print("Uploading weights to DagsHub Artifacts...")
    # Log BOTH files inside a single artifact directory named "weights"
    mlflow.log_artifact(local_weights_path, artifact_path="weights")
    mlflow.log_artifact(last_weights_path, artifact_path="weights")
    
    print("Registering model in the DagsHub Model Registry...")
    # Create an MLflow client to bypass strict flavor requirements
    client = MlflowClient()
    
    # Check if the registered model container already exists; if not, create it
    try:
        client.get_registered_model("Soccer-Tracking")
    except Exception:
        print("Creating new registered model container 'Soccer-Tracking'...")
        client.create_registered_model("Soccer-Tracking")
        
    # Directly force a new version from our raw artifact directory path
    source_uri = f"runs:/{run.info.run_id}/weights"
    model_version = client.create_model_version(
        name="Soccer-Tracking",
        source=source_uri,
        run_id=run.info.run_id
    )

print(f"Successfully logged and registered Version {model_version.version} of your model!")

Initialized MLflow to track repo "AgabaEmbedded/Soccer-Tracking"

Repository AgabaEmbedded/Soccer-Tracking initialized!

Uploading weights to DagsHub Artifacts...
Registering model in the DagsHub Model Registry...


2026/09/15 11:43:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Soccer-Tracking, version 5


🏃 View run model_upload at: https://dagshub.com/AgabaEmbedded/Soccer-Tracking.mlflow/#/experiments/4/runs/3b501f9ef00a4c6a82975c3fb28fe4eb
🧪 View experiment at: https://dagshub.com/AgabaEmbedded/Soccer-Tracking.mlflow/#/experiments/4
Successfully logged and registered Version 5 of your model!


In [8]:
!yolo val model=/kaggle/working/runs/detect/Large-Images-3Classes/weights/best.pt data=/kaggle/working/Soccer-Tracking-Large-2/data.yaml split=val imgsz=1088,920 batch=16

WARNING ⚠️ updating to 'imgsz=1088'. 'train' and 'val' imgsz must be an integer, while 'predict' and 'export' imgsz may be a [h, w] list or an integer, i.e. 'yolo export imgsz=640,480' or 'yolo export imgsz=640'
Ultralytics 8.4.152 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,351,765 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2904.5±342.4 MB/s, size: 243.5 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-2/valid/labels.cache... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 155.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 1.1it/s 47.3s
                   all        855      13852       0.96      0.911      0.959      0.629
            goalkeeper        480        480      0.967      0.857      0.933      0.616
                player        855      12121       0.96      0.963      

In [9]:
!yolo val model=/kaggle/working/runs/detect/Large-Images-3Classes/weights/last.pt data=/kaggle/working/Soccer-Tracking-Large-2/data.yaml split=val imgsz=1088,920 batch=16

WARNING ⚠️ updating to 'imgsz=1088'. 'train' and 'val' imgsz must be an integer, while 'predict' and 'export' imgsz may be a [h, w] list or an integer, i.e. 'yolo export imgsz=640,480' or 'yolo export imgsz=640'
Ultralytics 8.4.152 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,351,765 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2912.7±315.8 MB/s, size: 227.1 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-2/valid/labels.cache... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 163.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 1.1it/s 47.2s
                   all        855      13852      0.953      0.918      0.962      0.629
            goalkeeper        480        480      0.956      0.868      0.934      0.617
                player        855      12121      0.954      0.966      

In [10]:
!yolo val model=/kaggle/working/runs/detect/Large-Images-3Classes/weights/best.pt data=/kaggle/working/Soccer-Tracking-Large-2/data.yaml split=test imgsz=1088,920 batch=16

WARNING ⚠️ updating to 'imgsz=1088'. 'train' and 'val' imgsz must be an integer, while 'predict' and 'export' imgsz may be a [h, w] list or an integer, i.e. 'yolo export imgsz=640,480' or 'yolo export imgsz=640'
Ultralytics 8.4.152 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,351,765 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2742.9±718.1 MB/s, size: 226.1 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-2/test/labels... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 1.0Kit/s 0.8s
val: New cache created: /kaggle/working/Soccer-Tracking-Large-2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 1.1it/s 47.2s
                   all        855      13751      0.959      0.916      0.969       0.63
            goalkeeper        460        460      0.961      0.863      0.953      0.625

In [11]:
!yolo val model=/kaggle/working/runs/detect/Large-Images-3Classes/weights/last.pt data=/kaggle/working/Soccer-Tracking-Large-2/data.yaml split=test imgsz=1088,920 batch=16

WARNING ⚠️ updating to 'imgsz=1088'. 'train' and 'val' imgsz must be an integer, while 'predict' and 'export' imgsz may be a [h, w] list or an integer, i.e. 'yolo export imgsz=640,480' or 'yolo export imgsz=640'
Ultralytics 8.4.152 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26m summary (fused): 130 layers, 20,351,765 parameters, 0 gradients, 68.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2318.2±399.6 MB/s, size: 272.9 KB)
val: Scanning /kaggle/working/Soccer-Tracking-Large-2/test/labels.cache... 855 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 855/855 170.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 1.1it/s 47.7s
                   all        855      13751      0.964      0.917       0.97      0.629
            goalkeeper        460        460      0.966      0.871      0.955      0.627
                player        855      12061       0.96      0.962      0